### Data Preprocessing

#### Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
reviews = pd.read_csv("rotten_tomatoes_movie_reviews.csv")
movies = pd.read_csv("rotten_tomatoes_movies.csv")

: 

#### Parse

In [ ]:
# We will start parsing by removing duplicate movies and reviews
movies = movies.drop_duplicates(subset='id')

# We turn Box Office sales from strings with letters (e.g 'M' for million) into dollar amounts
movies['boxOffice'] = movies['boxOffice'].str.replace('$', '').str.replace('M', 'e6').str.replace('K', 'e3')
movies['boxOffice'] = pd.to_numeric(movies['boxOffice'], errors='coerce')

# Now we convert release dates and review dates from text to datetime
movies['releaseDateTheaters'] = pd.to_datetime(movies['releaseDateTheaters'], errors='coerce')
movies['releaseDateStreaming'] = pd.to_datetime(movies['releaseDateStreaming'], errors='coerce')

# The genre can be a list of multiple genres, so we take the first one as the movie’s main genre
movies['primaryGenre'] = movies['genre'].str.split(',').str[0]

# Define the difference in scores
movies['scoreGap'] = movies['audienceScore'] - movies['tomatoMeter']

movies[['title', 'tomatoMeter', 'audienceScore', 'scoreGap', 'primaryGenre', 'boxOffice']].head()

In [ ]:
# Removing duplicates
reviews = reviews.drop_duplicates(subset='reviewId')
reviews['creationDate'] = pd.to_datetime(reviews['creationDate'], errors='coerce')

# In the raw data, the critics' original scores are in different formats such as ("3/4", "8/10", "B+")
# We can convert the fraction ones to a [0-1] scale so they can be compared (other formats become NaN)
parts = reviews['originalScore'].str.split('/', expand=True)
reviews['scoreNorm'] = pd.to_numeric(parts[0], errors='coerce') / pd.to_numeric(parts[1], errors='coerce')
reviews.loc[reviews['scoreNorm'] > 1, 'scoreNorm'] = np.nan

reviews['isFresh'] = reviews['reviewState'] == 'fresh'

reviews[['originalScore', 'scoreNorm', 'reviewState', 'isFresh']].head()

#### Organize

In [ ]:
# Count each movie's reviews and its percentage of Fresh reviews
review_counts = reviews.groupby('id').agg(numReviews=('reviewId', 'count'), freshRate=('isFresh', 'mean')).reset_index()
movies = movies.merge(review_counts, on='id', how='left')
display(movies.head())

# Keep movies with both an audience and Tomatometer (critic) score
scored = movies.dropna(subset=['tomatoMeter', 'audienceScore'])
scored[['tomatoMeter', 'audienceScore', 'scoreGap', 'runtimeMinutes', 'numReviews']].describe()

### Data Exploration

## Hypothesis 1: Is there a difference between audience and critic scores?

> H0: There is no difference between audience and critic scores. The mean of (audience score - critic score) is 0.

> HA: There is a difference between audience and critic scores. The mean of (audience score - critic score) is not 0.

Use a **paired t-test** because every movie has both an audience score and a critic score, so the two samples are paired and not independent. Our sample is very large, so by the CLT the mean difference is approximately normal.

In [ ]:
t_stat, p = ttest_rel(scored['audienceScore'], scored['tomatoMeter'])
print("mean critic score:", scored['tomatoMeter'].mean())
print("mean audience score:", scored['audienceScore'].mean())
print("mean gap:", scored['scoreGap'].mean())
print("t-statistic:", t_stat)
print("p-value:", p)

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(scored['scoreGap'], bins=50, color='orange', edgecolor='black')
plt.axvline(0, color='black', linestyle='--', label='No difference')
plt.axvline(scored['scoreGap'].mean(), color='red', label='Mean gap')
plt.xlabel('Audience Score - Critic Score')
plt.ylabel('Number of Movies')
plt.title('Difference Between Audience and Critic Scores')
plt.legend()
plt.show()

Conclusion: The p-value is 1.22e-63, which is less than 0.05, so we reject the null hypothesis. There is a significant difference between audience and critic scores. On average, critics score movies 2.2 points higher than audiences (64.5 vs. 62.3).

## Hypothesis 2: Is the gap wider in certain genres?

> H0: Genre has no effect on the gap. The mean score gap is the same for all genres.

> HA: Genre does have an effect on the gap. At least one genre has a different mean score gap.

Use a **one-way ANOVA** because we're comparing the mean of one numeric variable (score gap) across more than 2 groups. We use each movie's first genre so no movie is counted twice, and only the 6 most common genres so each group has enough movies.

In [ ]:
top_genres = scored['primaryGenre'].value_counts().head(6).index
groups = [scored[scored['primaryGenre'] == g]['scoreGap'] for g in top_genres]

f_stat, p = f_oneway(*groups)
print(scored[scored['primaryGenre'].isin(top_genres)].groupby('primaryGenre')['scoreGap'].mean())
print("\nF-statistic:", f_stat)
print("p-value:", p)

In [ ]:
plt.figure(figsize=(10, 5))
plt.boxplot(groups, tick_labels=top_genres, showfliers=False)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Genre')
plt.ylabel('Audience Score - Critic Score')
plt.title('Score Gap by Genre')
plt.show()